# Grid build — regional half-hourly grid carbon (and optionally prices)

The Data Centre Optimiser runs this notebook, through `backend.runner.run_lca`,
whenever a site needs a grid series that is not in `custom_grid_lca_outputs/`
yet (`optimisation.sites.build_grid_series`). It patches `dashboard_config.py`
with the region and window, runs this, and restores the file afterwards.

It only orchestrates. The work is in the child notebooks, which run in this
kernel so they can share state (`5.prices.ipynb` reads `DASHBOARD_GRID_RUN_ROWS`,
which `3.1` leaves behind):

1. `2.tech_lca_foreground.ipynb` — only when the foreground databases are missing
2. `3.1.custom_grid_carbon_intensity_api.ipynb` — Carbon Intensity API mix → kg CO2e/kWh per half-hour
3. `5.prices.ipynb` — Elexon wholesale price over the same window, when asked

Replaces `1.dashboard_lca_adaptive.ipynb`, which also ran the electrolyser
wind/grid LCA (`4.wind_power.ipynb`) and its plots on every build.

In [ ]:
import dashboard_config as cfg
import lca_helpers as H  # noqa: F401 — fail here, not three notebooks in

cfg.print_dashboard()

In [ ]:
RUN_FOREGROUND_NOTEBOOK = bool(getattr(cfg, "RUN_FOREGROUND_NOTEBOOK_FROM_DASHBOARD", False))
RUN_GRID_NOTEBOOK       = bool(getattr(cfg, "RUN_GRID_NOTEBOOK_FROM_DASHBOARD", True))
RUN_PRICE_NOTEBOOK      = bool(getattr(cfg, "RUN_PRICE_NOTEBOOK_FROM_DASHBOARD", False))

print("Grid build plan")
print("---------------")
print("Foreground notebook:", RUN_FOREGROUND_NOTEBOOK)
print("Grid notebook:      ", RUN_GRID_NOTEBOOK)
print("Price notebook:     ", RUN_PRICE_NOTEBOOK)

if RUN_FOREGROUND_NOTEBOOK:
    get_ipython().run_line_magic("run", '"2.tech_lca_foreground.ipynb"')
if RUN_GRID_NOTEBOOK:
    get_ipython().run_line_magic("run", '"3.1.custom_grid_carbon_intensity_api.ipynb"')
if RUN_PRICE_NOTEBOOK:
    get_ipython().run_line_magic("run", '"5.prices.ipynb"')